# Tutorial 06 - Project: Iris Exploratory Data Analysis

The classical closing exercise of the course: one dataset, four numeric
measurements, three species - and every step of an exploratory analysis done
in order. Loads sklearn's bundled copy of Fisher's iris data (no internet
needed), checks structure and missingness, profiles each species, and then
asks the two questions the dataset exists to answer: which measurement
separates the species, and where exactly do two of them stop being separable.
Every figure is audited in memory so a blank render fails the notebook.

---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/09_Iris_Exploratory_Data_Analytics/` project folder
(anchored on its `README.md`; the notebook writes nothing to disk) by walking
upward from the current working directory - so the notebook runs identically
from the repository root, the project folder or its `notebooks/` sub-folder.
On Google Colab the repository is cloned into `/content/APSD_TAI_LAB`
automatically (opening the notebook directly from GitHub also works). The
resolved project folder is added to `sys.path` so its local modules import
cleanly.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "09_Iris_Exploratory_Data_Analysis"
ANCHOR_REL = "README.md"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")

### 1. Load the Dataset and Check its Structure

sklearn bundles the iris measurements locally (`sklearn.datasets.load_iris`),
so the notebook needs no internet and writes no files - unlike the seaborn
`load_dataset` route used in project 08. The structure check is the first
exploration step: 150 rows, 4 numeric features, 3 species at exactly 50 rows
each (a balanced design, which is why no class weighting appears anywhere
later), zero missing values.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.rename(
    columns={c: c.replace(" (cm)", "").replace(" ", "_") for c in iris.feature_names}
)
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
feats = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

print("shape:", df.shape, "| missing:", int(df.isna().sum().sum()))
print("rows per species:", df["species"].value_counts().to_dict())
df[feats].describe().round(2)

### 2. Per-Species Profiles

The group means show the ordering every later chart will echo: setosa is the
small-petal species (1.46 cm petal length, 0.25 cm petal width) and virginica
the large one (5.55 / 2.03), with versicolor in between. Sepals do not follow
the same ranking - setosa actually has the *widest* sepals on average (3.43 cm) -
which is the first sign that petal dimensions carry the discrimination.

In [ ]:
profile = df.groupby("species")[feats].agg(["mean", "std"]).round(2)
display(profile)

means = df.groupby("species")[feats].mean().round(2)
display(means)
print("petal length range per species:")
display(df.groupby("species")["petal_length"].agg(["min", "max"]))

### 3. Figure Audit Helper

Every figure below is rendered to an in-memory PNG and measured (non-blank
pixel ratio, distinct colours, byte size), printed and finally asserted in
Section 9 - the same audit pattern as the project 06 charts, so a blank or
data-less render cannot pass silently. Nothing is written to the repository:
figures are closed right after auditing.

In [ ]:
import io

from PIL import Image

AUDITS = []

def audit(fig, name):
    """Render fig to an in-memory PNG, measure it, record the audit, close it."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    pixels = np.asarray(image)
    non_white = (pixels < 250).any(axis=2).mean()
    colors = {tuple(c) for row in pixels[::4] for c in row[::4]}
    AUDITS.append((name, non_white, len(colors), buf.getbuffer().nbytes))
    plt.close(fig)
    print(f"{name:30s} non-blank {non_white:6.1%} | distinct colors {len(colors):4d}")
    return fig

### 4. Correlation Heatmap - and the Setosa Twist

The all-rows correlation matrix says petal length and petal width move
together (0.96) and that sepal width is *negatively* related to everything
else (-0.43 with petal length). That second reading is an artefact of
pooling: within setosa alone the sepal correlation flips to +0.74, because
each species sits in its own corner of the space and the pooled line runs
through the gaps between clusters, not through any within-species relation.
The notebook prints both matrices next to the heatmap - Simpson's paradox is
the standard name for this flip, and iris is its textbook example.

In [ ]:
corr = df[feats].corr().round(2)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix (all 150 rows)", fontweight="bold")
audit(fig, "correlation heatmap (pooled)")
display(corr)

setosa_corr = df[df["species"] == "setosa"][feats].corr().round(2)
print("within-setosa correlations (sepal length vs sepal width):",
      setosa_corr.loc["sepal_length", "sepal_width"],
      "| pooled:", corr.loc["sepal_length", "sepal_width"])

### 5. The Setosa Rule - Petal Length Separates it Perfectly

The ranges found in Section 2 become a rule: setosa's longest petal is 1.9 cm
and the shortest petal of any other species is 3.0 cm. A one-line rule -
`petal_length < 2.5` - classifies setosa with zero errors on all 150 rows.
The bar chart of petal length by species with the threshold drawn shows the
empty gap the rule lives in; the cell prints the rule's accuracy.

In [ ]:
setosa_max = df.loc[df["species"] == "setosa", "petal_length"].max()
others_min = df.loc[df["species"] != "setosa", "petal_length"].min()
print(f"setosa max petal length: {setosa_max} | min among the other species: {others_min}")

rule = (df["petal_length"] < 2.5)
accuracy = (rule == (df["species"] == "setosa")).mean()
print(f"rule 'petal_length < 2.5 cm -> setosa' accuracy: {accuracy:.0%}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.barplot(data=df, x="species", y="petal_length", hue="species", errorbar="sd",
            palette="Set2", ax=ax)
ax.axhline(2.5, color="black", linestyle="--", linewidth=1.5, label="rule threshold 2.5")
ax.set_title("Mean Petal Length per Species (+-1 SD) with the Setosa Rule Threshold", fontweight="bold")
ax.set_xlabel("Species")
ax.set_ylabel("Petal length (cm)")
ax.legend()
audit(fig, "setosa rule bar")

### 6. The Petal Scatter - and Where Separation Ends

Petal length against petal width, one colour per species. Setosa floats free
in its own corner; the other two overlap. The overlap is narrow but real:
versicolor's longest petals reach 5.1 cm while virginica's shortest are
4.5 cm, so a 0.6 cm band exists where both species live - the zone the
dashed rectangle marks, and the exact reason iris is not solvable by one
threshold on two measurements. Any further separation must come from a model
that combines features, not from a sharper cutoff.

In [ ]:
v_max = df.loc[df["species"] == "versicolor", "petal_length"].max()
g_min = df.loc[df["species"] == "virginica", "petal_length"].min()
print(f"versicolor petal-length max: {v_max} | virginica min: {g_min} | overlap zone: {v_max - g_min:.1f} cm")

fig, ax = plt.subplots(figsize=(7.5, 5.5))
sns.scatterplot(data=df, x="petal_length", y="petal_width", hue="species",
                style="species", s=70, palette="Set2", ax=ax)
ax.add_patch(plt.Rectangle((g_min - 0.05, 1.0), v_max - g_min + 0.1, 0.9,
                           fill=False, edgecolor="red", linestyle="--", linewidth=1.6))
ax.annotate("overlap zone", xy=((v_max + g_min) / 2, 1.45), color="red",
            fontsize=10, ha="center")
ax.set_title("Petal Length vs Petal Width (setosa free, versicolor x virginica overlap marked)", fontweight="bold")
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
audit(fig, "petal scatter + overlap")

### 7. Distribution Comparison - which Measurement Separates Best

The four distribution plots, one per measurement, make the ranking visible:
petal length and petal width separate the species into three clearly spaced
distributions; sepal width is the weakest - its species distributions sit
almost on top of each other, and the ordering even differs from the petal
ranking (setosa is the *largest* on sepal width). The box plots of petal
length quantify the same gap per species.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
for ax, feat in zip(axs, feats):
    sns.kdeplot(data=df, x=feat, hue="species", fill=True, common_norm=False,
                palette="Set2", alpha=0.5, ax=ax)
    ax.set_title(feat.replace("_", " "), fontsize=11)
fig.suptitle("Per-Measurement Species Distributions", fontweight="bold")
audit(fig, "4x distribution comparison")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.boxplot(data=df, x="species", y="petal_length", hue="species", palette="Set2", ax=ax)
ax.set_title("Petal Length by Species (box plot)", fontweight="bold")
ax.set_xlabel("Species")
ax.set_ylabel("Petal length (cm)")
audit(fig, "petal length boxplot")

### 8. Pair Grid - the Whole Space at a Glance

`pairplot` draws every pair of measurements with the species coloured, which
is the complete map: setosa is separable on any petal panel, and the
versicolor/virginica tangle is present on every panel that involves petals.
Sepal panels alone would not separate setosa cleanly from the others on the
width axis - the pair grid is what shows the petal pair is the best projection
without computing anything.

In [ ]:
grid = sns.pairplot(df, vars=feats, hue="species", palette="Set2",
                    diag_kind="kde", height=1.9)
grid.figure.suptitle("Iris Pairwise Relationships by Species", y=1.02, fontweight="bold")
audit(grid.figure, "pairplot grid")

### 9. Audit Summary and Conclusions

All seven figures audited, then the conclusions the exercise asks for: which
measurement discriminates (petal length, with petal width a close second - the
pair 0.96-correlated), which species is trivially separable (setosa, by the
2.5 cm petal rule with 100% accuracy on this dataset), and where the problem
actually lives (a 0.6 cm petal-length band where versicolor and virginica
overlap, visible on every petal panel of the pair grid). The pooled
correlations are pooled artefacts - within one species the sepal relation
flips sign - so any model built next should carry species-aware features
rather than the pooled numbers.

In [ ]:
audit_table = pd.DataFrame(AUDITS, columns=["figure", "non_blank_ratio", "distinct_colors", "png_bytes"])
audit_table["non_blank_ratio"] = audit_table["non_blank_ratio"].round(3)
display(audit_table)

failed = audit_table[(audit_table["non_blank_ratio"] < 0.01) | (audit_table["distinct_colors"] < 10)]
assert failed.empty, f"blank or empty figure(s): {failed['figure'].tolist()}"
print(f"all {len(AUDITS)} figures drawn, audited non-blank and closed")

print("petal length x petal width correlation (pooled):", round(df.petal_length.corr(df.petal_width), 3))
print("sepal length x sepal width correlation (pooled):", round(df.sepal_length.corr(df.sepal_width), 3))